# Aula 2 — Treinando Redes Neurais Profundas (PyTorch)

Notebook de acompanhamento da Aula 2 — **Tópicos Avançados em IA · UFABC**  
Adaptado de MIT 15.773 (Farias & Ramakrishnan, 2024)

---

## Índice

1. [Configuração](#1-configuração)
2. [Parte 1 — Dataset: Doença Cardíaca](#2-parte-1--dataset-doença-cardíaca)
3. [Parte 2 — Definindo o Modelo](#3-parte-2--definindo-o-modelo)
4. [Parte 3 — Loss e Otimizador](#4-parte-3--loss-e-otimizador)
5. [Parte 4 — Autograd: gradientes automáticos](#5-parte-4--autograd-gradientes-automáticos)
6. [Parte 5 — Treinamento com Early Stopping manual](#6-parte-5--treinamento-com-early-stopping-manual)
7. [Parte 6 — Avaliação](#7-parte-6--avaliação)
8. [Parte 7 — Fashion-MNIST (Multiclasse)](#8-parte-7--fashion-mnist-multiclasse)

## 1. Configuração

In [ ]:
# Instala dependências (necessário no Colab)
!pip install -q ucimlrepo torch torchvision

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando device: {device}')

## 2. Parte 1 — Dataset: Doença Cardíaca

Usamos o **Cleveland Clinic Heart Disease Dataset** (UCI ML Repository, id=45):  
~300 pacientes, 13 atributos clínicos → saída binária (doença: sim/não).

Após *one-hot encoding* das variáveis categóricas, chegamos a **~29 entradas**.

In [ ]:
from ucimlrepo import fetch_ucirepo

heart = fetch_ucirepo(id=45)
X_raw = heart.data.features
y_raw = heart.data.targets

print(f"Exemplos: {X_raw.shape[0]}  |  Features brutas: {X_raw.shape[1]}")
X_raw.head()

In [ ]:
# Alvo: 0 = sem doença, 1 = com doença (qualquer grau > 0)
y = (y_raw.values.ravel() > 0).astype(np.float32)

# Preenche valores ausentes com a mediana
X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

# One-hot encoding das colunas categóricas
cat_cols = ['cp', 'restecg', 'slope', 'thal']
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=False)

print(f"Features após one-hot: {X.shape[1]}")
print(f"Proporção de casos positivos: {y.mean():.1%}")

In [ ]:
# Split treino / validação / teste  (70 / 15 / 15)
X_train_np, X_temp, y_train_np, y_temp = train_test_split(
    X.values.astype(np.float32), y, test_size=0.30, random_state=42, stratify=y
)
X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Normalização
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

# Converte para tensores PyTorch
def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.float32).to(device),
            torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(device))

X_train, y_train = to_tensor(X_train_np, y_train_np)
X_val,   y_val   = to_tensor(X_val_np,   y_val_np)
X_test,  y_test  = to_tensor(X_test_np,  y_test_np)

n_features = X_train.shape[1]
print(f"Treino: {X_train.shape}  |  Val: {X_val.shape}  |  Teste: {X_test.shape}")
print(f"Número de features (entradas da rede): {n_features}")

## 3. Parte 2 — Definindo o Modelo

Arquitetura: **entrada → 16 ReLU → dropout → 1 sigmoide**

Em PyTorch usamos `nn.Sequential` para empilhar camadas.

In [ ]:
model = nn.Sequential(
    nn.Linear(n_features, 16),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(16, 1),
    nn.Sigmoid(),
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal de parâmetros: {total_params}")

## 4. Parte 3 — Loss e Otimizador

Em PyTorch, loss e otimizador são instanciados separadamente — ao contrário do `model.compile()` do Keras.

| Tipo de saída | Loss PyTorch |
|---|---|
| Binária (sigmoide) | `nn.BCELoss()` |
| Multiclasse (logits) | `nn.CrossEntropyLoss()` — inclui softmax internamente |
| Regressão | `nn.MSELoss()` |

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 5. Parte 4 — Autograd: gradientes automáticos

O PyTorch constrói um **grafo computacional** durante o forward pass.  
Chamar `.backward()` percorre esse grafo de trás para frente e acumula os gradientes em `.grad`.

In [ ]:
# Exemplo simples do slide: z = w*x + b, loss = (z - y)^2
x_eg   = torch.tensor([2.0], requires_grad=True)
w_eg   = torch.tensor([0.5], requires_grad=True)
b_eg   = torch.tensor([1.0], requires_grad=True)
y_real = torch.tensor([4.0])

# forward — grafo é construído aqui
yh   = w_eg * x_eg + b_eg
loss_eg = (yh - y_real) ** 2

# backward — percorre o grafo automaticamente
loss_eg.backward()

print(f"∂loss/∂w = {w_eg.grad.item():.1f}  (esperado: -20.0)")
print(f"∂loss/∂b = {b_eg.grad.item():.1f}  (esperado: -10.0)")

## 6. Parte 5 — Treinamento com Early Stopping manual

O laço explícito do PyTorch deixa cada operação visível:

```
zero_grad → forward → backward → step
```

Early stopping é implementado manualmente: guardamos os pesos do melhor momento e paramos quando a paciência se esgota.

In [ ]:
import copy

# Hiper-parâmetros
epochs   = 200
patience = 10

best_val_loss = float('inf')
wait          = 0
best_weights  = None
history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(epochs):
    # ── Treino ──────────────────────────────────────────
    model.train()
    optimizer.zero_grad()
    train_loss = criterion(model(X_train), y_train)
    train_loss.backward()
    optimizer.step()

    # ── Validação ───────────────────────────────────────
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val)
        val_loss  = criterion(val_preds, y_val).item()

        train_acc = ((model(X_train) >= 0.5).float() == y_train).float().mean().item()
        val_acc   = ((val_preds      >= 0.5).float() == y_val  ).float().mean().item()

    history['train_loss'].append(train_loss.item())
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # ── Early Stopping ──────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait          = 0
        best_weights  = copy.deepcopy(model.state_dict())
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping na época {epoch + 1}")
            break

model.load_state_dict(best_weights)
print(f"Melhor val_loss: {best_val_loss:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Treino')
axes[0].plot(history['val_loss'],   label='Validação')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Treino')
axes[1].plot(history['val_acc'],   label='Validação')
axes[1].set_title('Acurácia')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Parte 6 — Avaliação

In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test)
    test_loss  = criterion(test_preds, y_test).item()
    y_pred_bin = (test_preds >= 0.5).float()
    test_acc   = (y_pred_bin == y_test).float().mean().item()

print(f"Loss no teste : {test_loss:.4f}")
print(f"Acurácia no teste: {test_acc:.1%}")

In [ ]:
y_pred_np = y_pred_bin.cpu().numpy().ravel().astype(int)
y_test_np_cpu = y_test.cpu().numpy().ravel().astype(int)

print(classification_report(y_test_np_cpu, y_pred_np, target_names=['Sem doença', 'Com doença']))

ConfusionMatrixDisplay(
    confusion_matrix(y_test_np_cpu, y_pred_np),
    display_labels=['Sem doença', 'Com doença']
).plot(cmap='Blues')
plt.title('Matriz de Confusão — Conjunto de Teste')
plt.show()

## 8. Parte 7 — Fashion-MNIST (Multiclasse)

Classificação de 10 categorias de roupas a partir de imagens 28×28.

Diferença-chave em relação ao Keras:
- `nn.CrossEntropyLoss()` **já inclui o softmax internamente** — **não** adicione `nn.Softmax` ao modelo.
- Usamos `DataLoader` para iterar em mini-batches.

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([transforms.ToTensor()])

train_ds = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

class_names = ['T-shirt','Calça','Pulôver','Vestido','Casaco',
               'Sandália','Camisa','Tênis','Bolsa','Bota']

# Visualiza exemplos
imgs, lbls = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    ax.set_title(class_names[lbls[i].item()])
    ax.axis('off')
plt.suptitle('Amostras do Fashion-MNIST')
plt.tight_layout()
plt.show()

In [ ]:
model_f = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10),   # logits — CrossEntropyLoss aplica softmax internamente
).to(device)

criterion_f = nn.CrossEntropyLoss()
optimizer_f = optim.Adam(model_f.parameters(), lr=1e-3)

print(model_f)

In [ ]:
n_epochs_f = 15
hist_f     = {'train_loss': [], 'val_acc': []}

for epoch in range(n_epochs_f):
    # ── Treino ──────────────────────────────────────────
    model_f.train()
    running_loss = 0.0
    for imgs_b, lbls_b in train_loader:
        imgs_b, lbls_b = imgs_b.to(device), lbls_b.to(device)
        optimizer_f.zero_grad()
        loss_b = criterion_f(model_f(imgs_b), lbls_b)
        loss_b.backward()
        optimizer_f.step()
        running_loss += loss_b.item()

    # ── Validação ───────────────────────────────────────
    model_f.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs_b, lbls_b in test_loader:
            imgs_b, lbls_b = imgs_b.to(device), lbls_b.to(device)
            preds = model_f(imgs_b).argmax(dim=1)
            correct += (preds == lbls_b).sum().item()
            total   += lbls_b.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc  = correct / total
    hist_f['train_loss'].append(epoch_loss)
    hist_f['val_acc'].append(epoch_acc)

    if (epoch + 1) % 5 == 0:
        print(f"Época {epoch+1:3d}/{n_epochs_f} — loss: {epoch_loss:.4f}  acc teste: {epoch_acc:.1%}")

print(f"\nAcurácia final no teste (Fashion-MNIST): {hist_f['val_acc'][-1]:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_f['train_loss'])
axes[0].set_title('Loss de treino — Fashion-MNIST')
axes[0].set_xlabel('Época')

axes[1].plot(hist_f['val_acc'])
axes[1].set_title('Acurácia no teste — Fashion-MNIST')
axes[1].set_xlabel('Época')

plt.tight_layout()
plt.show()

In [ ]:
# Predições em exemplos do teste
model_f.eval()
sample_imgs, sample_lbls = next(iter(test_loader))
sample_imgs = sample_imgs[:10].to(device)

with torch.no_grad():
    preds_f = model_f(sample_imgs).argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(sample_imgs[i].squeeze().cpu(), cmap='gray')
    real  = sample_lbls[i].item()
    color = 'green' if preds_f[i] == real else 'red'
    ax.set_title(f"Pred: {class_names[preds_f[i]]}\nReal: {class_names[real]}",
                 color=color, fontsize=8)
    ax.axis('off')
plt.suptitle('Predições (verde = correto, vermelho = errado)')
plt.tight_layout()
plt.show()